# Transported separatrix-connectivity barycenter reconstruction

This notebook reconstructs MSC-like barycenter topology from a saved PFGW bundle.

**Method.** Infer topology from transported exact adjacency and transported separatrix-local connectivity. Use `separatrixID` as the authoritative source of chain provenance. Soft paths remain inside one fixed recorded separatrix ID. Overlapping separatrices are retained without merging identities. `C_center` is used only for optional post-hoc comparison after the graph and all reconstruction parameters are fixed.

This notebook does not rerun the PFGW solver and does not claim an exact Morse–Smale complex without a later scalar-field realization.


## Configuration

Defaults are deterministic exploration settings, not theoretically optimal values. Reconstruction artifacts are written under a new timestamped directory and do not overwrite previous reconstructions.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import Markdown, display

from lib.barycenter_io import load_barycenter_bundle
from lib.msc_barycenter_reconstruction_separatrix import (
    ReconstructionConfig,
    TERMINOLOGY_VERSION,
    validate_barycenter_bundle,
    decode_barycenter_nodes,
    compute_transported_separatrix_connectivity,
    construct_candidate_edge_table,
    infer_target_component_count,
    build_maximum_support_forest,
    augment_consensus_edges,
    local_repair_topology,
    validate_reconstructed_topology,
    run_q_hop_sensitivity,
    run_sampling_mode_sensitivity,
    compare_posthoc_baselines,
    build_evidence_density_field,
    route_graph_through_density,
    save_reconstruction_bundle,
    load_graph_pickle,
    verify_reloaded_topology,
    run_synthetic_separatrix_tests,
    graph_objective_components,
)

PROJECT_ROOT = Path(".").resolve()
config = ReconstructionConfig(
    run_name="latest",
    reconstruction_name="transported_separatrix_connectivity",
    hop_decay_q=0.5,
    max_soft_hops=4,
    enable_density_routing=False,
)
BUNDLE_DIR = config.resolve_bundle_dir(PROJECT_ROOT)
OUTPUT_DIR, FIGURES_DIR = config.resolve_output_dirs(PROJECT_ROOT, create=True)
pd.DataFrame(
    [{"Setting": k, "Resolved value": v} for k, v in config.resolved_dict(PROJECT_ROOT).items()]
).head(20)


## Terminology used in this notebook

| Term | Meaning |
| --- | --- |
| Direct-adjacency support | Conditional consistency of exact input adjacency |
| Separatrix-connectivity support | Conditional consistency of nearby connectivity along one fixed recorded separatrix |
| Pair coverage | Amount of weighted correspondence information available for both endpoints |
| Direct-adjacency evidence | Coverage-weighted exact adjacency evidence |
| Separatrix-connectivity evidence | Coverage-weighted separatrix-local evidence |
| Soft-extension evidence | Evidence contributed by nearby-but-not-direct pairs |
| Connectivity-selection score | Explicit score used to rank candidate barycenter edges |

$$
E_{ij}^{\mathrm{direct}} = O_{ij} S_{ij}^{\mathrm{direct}}, \qquad
E_{ij}^{\mathrm{sep}} = O_{ij} S_{ij}^{\mathrm{sep}}.
$$

- High conditional support with low coverage is consistent but scarce evidence.
- High coverage with low direct support is evidence against exact adjacency.
- High separatrix support with low direct support indicates likely sampling mismatch along a shared separatrix.
- Support must not be interpreted without coverage.

Shared-separatrix membership is **not** transitive. Soft affinity paths must retain one fixed separatrix ID:

```text
u -- v -- w
edge (u, v): separatrixID = "0"
edge (v, w): separatrixID = "1"
```

Then $\Sigma(u)=\{0\}$, $\Sigma(v)=\{0,1\}$, $\Sigma(w)=\{1\}$, so $u$ and $w$ receive zero separatrix-local affinity even though their unrestricted hop distance is two.

Hop decay is tolerant to small sampling mismatches but is not completely invariant to nonuniform sampling density.


## Load and validate the barycenter bundle

Aligned inputs `Ts[s]`, `Cs[s]`, `Ys[s]`, `ps[s]`, `lambdas[s]`, `nodes_list[s]`, `Gs[s]`, and `mesh_ids[s]` are required. If couplings, graphs, or trustworthy node orderings are unavailable, reconstruction stops. There is no silent `C_center` fallback. A held-out seed is excluded from reconstruction and may appear only as a labeled post-hoc baseline.


In [ ]:
bundle = load_barycenter_bundle(BUNDLE_DIR)
validation = validate_barycenter_bundle(bundle, strict_validation=config.strict_validation)
input_node_orders = validation.input_node_orders
print(
    f"Loaded {validation.n_inputs} inputs, n_center={validation.n_center}, "
    f"held_out_seed={validation.has_held_out_seed}, exclude_seed={validation.exclude_seed}"
)
for warning in validation.warnings:
    print("warning:", warning)
display(Markdown(f"`terminology_version = {TERMINOLOGY_VERSION}`"))


## Decode and inspect barycenter nodes


In [ ]:
decoded = decode_barycenter_nodes(bundle, config)
nodes_df = decoded.dataframe.copy()
display(nodes_df.head())
print(
    "coordinates:", decoded.has_coordinates,
    "scalar:", decoded.has_scalar,
    "types:", decoded.has_types,
)
print(
    "effective constraints:",
    decoded.effective_type_constraint_mode,
    decoded.effective_scalar_order_constraint_mode,
    decoded.effective_planarity_mode,
)
nodes_df.to_csv(OUTPUT_DIR / "decoded_nodes.csv", index=False)


## Validate recorded separatrix annotations

Every input edge `separatrixID` is parsed as a set of IDs. A value such as `"0,1,3"` means the edge belongs independently to separatrices `"0"`, `"1"`, and `"3"`.


In [ ]:
# Annotation diagnostics are computed during transport; run a lightweight preview first.
from lib.msc_barycenter_reconstruction_separatrix import (
    build_separatrix_membership,
    diagnose_separatrix_annotations,
)

preview_rows = []
for s, graph in enumerate(bundle["Gs"]):
    membership = build_separatrix_membership(
        graph,
        attribute=config.separatrix_id_attribute,
        delimiter=config.separatrix_id_delimiter,
        strict=config.strict_separatrix_validation,
        missing_policy=config.missing_separatrix_id_policy,
    )
    annotation, subgraph_df, _, _, sampling = diagnose_separatrix_annotations(
        graph, s, membership
    )
    preview_rows.append(annotation)
preview_df = pd.DataFrame(preview_rows)
display(preview_df)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(preview_df["unique_separatrix_id_count"], bins=10, color="steelblue")
axes[0].set_title("Unique separatrix IDs / input")
axes[1].hist(preview_df["multi_id_edge_fraction"], bins=10, color="darkorange")
axes[1].set_title("Multi-ID edge fraction")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "separatrix_annotation_preview.png", dpi=140)
plt.show()


## Build direct adjacency and separatrix-local affinities

Direct adjacency is exact binary adjacency. Separatrix-local connectivity affinity uses hop decay
$K(u,v)=q^{d_{\mathrm{sep}}(u,v)-1}$ for $1 \le d_{\mathrm{sep}} \le H$ inside one fixed separatrix ID.


## Transport connectivity through PFGW couplings

Conditional supports answer different questions from pair coverage. Seed graphs held out of the barycenter are never included.


In [ ]:
transported = compute_transported_separatrix_connectivity(
    bundle, input_node_orders, config
)
for warning in transported.warnings:
    print("warning:", warning)
display(pd.Series(transported.diagnostics, name="value").to_frame())

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
matrices = [
    ("direct support", transported.conditional_direct_adjacency_support),
    ("separatrix support", transported.conditional_separatrix_connectivity_support),
    ("soft extension", transported.conditional_soft_extension_support),
    ("pair coverage", transported.pair_coverage),
    ("direct evidence", transported.direct_adjacency_evidence),
    ("separatrix evidence", transported.separatrix_connectivity_evidence),
]
for ax, (title, matrix) in zip(axes.ravel(), matrices):
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="viridis")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "transported_connectivity_heatmaps.png", dpi=140)
plt.show()

upper = np.triu(transported.pair_is_covered, 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(
    transported.conditional_direct_adjacency_support[upper],
    transported.conditional_separatrix_connectivity_support[upper],
    s=10,
    alpha=0.6,
)
axes[0].set_xlabel("direct support")
axes[0].set_ylabel("separatrix support")
axes[1].scatter(
    transported.pair_coverage[upper],
    transported.separatrix_connectivity_evidence[upper],
    s=10,
    alpha=0.6,
)
axes[1].set_xlabel("pair coverage")
axes[1].set_ylabel("separatrix evidence")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "support_coverage_scatter.png", dpi=140)
plt.show()


## Construct the barycenter candidate-edge table

Candidates come only from transported connectivity evidence. `barycenter_relation_distance` is stored for audit and does not enter ranking.


In [ ]:
C_center_audit = np.asarray(bundle["C_center"], dtype=float)
candidate_result = construct_candidate_edge_table(
    transported,
    nodes_df,
    config,
    barycenter_relation=C_center_audit,
)
candidates = candidate_result.selected_edges.copy()
display(pd.Series(candidate_result.diagnostics).to_frame("value"))
display(candidates.head())
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(
    candidates["direct_adjacency_evidence"],
    candidates["separatrix_connectivity_evidence"],
    s=12,
    alpha=0.7,
)
ax.set_xlabel("direct_adjacency_evidence")
ax.set_ylabel("separatrix_connectivity_evidence")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "candidate_evidence_scatter.png", dpi=140)
plt.show()


## Initialize a maximum-support skeleton forest

Replace metric-spanner construction with a constrained Kruskal-style maximum spanning forest over `connectivity_selection_score`. Do not use `C_center`.


In [ ]:
component_target = infer_target_component_count(bundle, config.target_component_count)
print(component_target)
forest_result = build_maximum_support_forest(
    candidates,
    nodes_df,
    config,
    component_target,
    effective_planarity_mode=decoded.effective_planarity_mode,
)
G_forest = forest_result.graph
display(pd.Series(forest_result.diagnostics).to_frame("value"))


## Add consensus edges and remove redundant soft edges

Soft affinity can create local cliques and chords. Soft-only edges that are explained by stronger two-step or selected-path routes are treated as potentially redundant. Strongly direct-supported edges are retained.


In [ ]:
augmented = augment_consensus_edges(
    G_forest,
    candidates,
    config,
    effective_planarity_mode=decoded.effective_planarity_mode,
)
repaired = local_repair_topology(
    augmented.graph,
    candidates,
    config,
    target_components=component_target.count,
    effective_planarity_mode=decoded.effective_planarity_mode,
)
G_topology = repaired.graph
print("added:", augmented.diagnostics)
print("repair objective:", repaired.objective_components)
display(repaired.edit_history.head())


## Validate the reconstructed topology

Every barycenter node is retained. Uncovered nodes stay isolated rather than being bridged by `C_center`.


In [ ]:
validation_result = validate_reconstructed_topology(
    G_topology,
    nodes_df,
    transported,
    candidates,
    config,
    effective_planarity_mode=decoded.effective_planarity_mode,
)
display(pd.Series(validation_result.metrics).to_frame("value"))
weak = validation_result.node_support_classes[
    validation_result.node_support_classes["support_class"].isin(
        ["weakly_supported", "uncovered"]
    )
]
display(weak)
from lib.msc_plotting import draw_msc_barycenter_graph
draw_msc_barycenter_graph(G_topology)
plt.gcf().savefig(FIGURES_DIR / "G_topology.png", dpi=140)
plt.show()


## Sampling-rate and parameter sensitivity

Evaluate `q in {0, 0.25, 0.5, 0.75, 1}` and `max_soft_hops in {2, 3, 4}`. Treat `q=1` as diagnostic only. Do not use `C_center` agreement to choose `(q, H)`. Configured defaults remain unless changed explicitly.

This grid re-runs transport and selection for each setting and may take several minutes on large supports.


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    sensitivity = run_q_hop_sensitivity(
        bundle,
        input_node_orders,
        nodes_df,
        config,
        effective_planarity_mode=decoded.effective_planarity_mode,
    )
    sampling_sensitivity = run_sampling_mode_sensitivity(
        bundle,
        input_node_orders,
        nodes_df,
        config,
        effective_planarity_mode=decoded.effective_planarity_mode,
    )
display(sensitivity)
display(sampling_sensitivity)
length_changed = False
if not sampling_sensitivity.empty and "changed_edge_count" in sampling_sensitivity:
    length_changed = bool(sampling_sensitivity["changed_edge_count"].iloc[0] > 0)


## Compare with post-hoc baselines

Baselines are evaluated only after reconstruction is fixed and do not alter the selected graph.


In [ ]:
# Direct-only q=0 reconstruction as a labeled baseline.
direct_config = ReconstructionConfig(
    run_name=config.run_name,
    reconstruction_name="direct_only_baseline",
    hop_decay_q=0.0,
    max_soft_hops=config.max_soft_hops,
    enable_length_normalized_affinity=False,
)
direct_transported = compute_transported_separatrix_connectivity(
    bundle, input_node_orders, direct_config, q=0.0
)
direct_candidates = construct_candidate_edge_table(
    direct_transported, nodes_df, direct_config
)
direct_forest = build_maximum_support_forest(
    direct_candidates.selected_edges,
    nodes_df,
    direct_config,
    component_target,
    effective_planarity_mode=decoded.effective_planarity_mode,
)
direct_aug = augment_consensus_edges(
    direct_forest.graph,
    direct_candidates.selected_edges,
    direct_config,
    effective_planarity_mode=decoded.effective_planarity_mode,
)
baseline_comparison = compare_posthoc_baselines(
    final_graph=G_topology,
    forest_graph=G_forest,
    nodes=nodes_df,
    transported=transported,
    bundle=bundle,
    config=config,
    direct_only_graph=direct_aug.graph,
)
display(baseline_comparison)


## Optional density-based geometric routing

> Density routing changes only the geometric realization of already selected edges. The node set and adjacency come entirely from transported input connectivity.

No Morse–Smale complex of the KDE field is computed here.


In [ ]:
routed_result = None
edge_weight_semantics = "euclidean_endpoint_length"
if config.enable_density_routing:
    selected_for_density = validation_result.selected_edges.copy()
    if selected_for_density.empty:
        raise RuntimeError("No selected edges available for density routing.")
    field = build_evidence_density_field(
        nodes_df,
        selected_for_density,
        config,
        resolution=config.density_grid_resolution_candidates[0],
        bandwidth=config.density_bandwidth_candidates[0],
        mass_column="connectivity_selection_score",
    )
    routed_result = route_graph_through_density(G_topology, nodes_df, field)
    G_topology = routed_result.graph
    edge_weight_semantics = "density_routed_polyline_length"
    display(routed_result.route_statistics.head())
else:
    display(Markdown("Density routing disabled (`enable_density_routing=False`)."))


## Post-hoc barycenter-relation consistency

`C_center` is audit-only. It must not generate candidates, rank edges, select `(q, H)`, build the forest, augment, repair, or set final graph weights.


In [ ]:
posthoc_relation_metrics = {"enabled": False}
if config.report_barycenter_relation_consistency:
    C = np.asarray(bundle["C_center"], dtype=float)
    graph_lengths = []
    relation_lengths = []
    for u, v, data in G_topology.edges(data=True):
        if np.isfinite(C[u, v]) and C[u, v] > 0:
            graph_lengths.append(float(data.get("weight", np.nan)))
            relation_lengths.append(float(C[u, v]))
    if graph_lengths:
        corr = float(np.corrcoef(graph_lengths, relation_lengths)[0, 1])
    else:
        corr = float("nan")
    posthoc_relation_metrics = {
        "enabled": True,
        "n_compared_edges": len(graph_lengths),
        "correlation_weight_vs_C_center": corr,
        "uses_barycenter_relation_for_edge_selection": False,
    }
display(posthoc_relation_metrics)


## Save and reload the reconstruction bundle


In [ ]:
warnings_and_fallbacks = list(validation.warnings) + list(transported.warnings)
manifest = save_reconstruction_bundle(
    OUTPUT_DIR,
    config=config,
    source_barycenter_dir=BUNDLE_DIR,
    source_metadata={
        **dict(bundle.get("meta") or {}),
        "exclude_seed": validation.exclude_seed,
    },
    nodes=nodes_df,
    transported=transported,
    candidates=candidate_result,
    forest=forest_result,
    augmented=augmented,
    repaired=repaired,
    validation=validation_result,
    sensitivity=sensitivity,
    sampling_sensitivity=sampling_sensitivity,
    baseline_comparison=baseline_comparison,
    routed=routed_result,
    posthoc_relation_metrics=posthoc_relation_metrics,
    warnings_and_fallbacks=warnings_and_fallbacks,
)
# Ensure required diagnostic CSVs exist even if save already wrote them.
transported.input_annotation_diagnostics.to_csv(
    OUTPUT_DIR / "separatrix_annotation_diagnostics.csv", index=False
)
display(Markdown(f"Saved reconstruction bundle to `{OUTPUT_DIR}`"))


### Reload verification


In [ ]:
G_reloaded = load_graph_pickle(OUTPUT_DIR / "G_topology.gpickle")
reload_report = verify_reloaded_topology(
    G_reloaded,
    expected_nodes=set(nodes_df["node_id"]),
    expected_edge_count=validation_result.metrics["edge_count"],
    saved_metrics=validation_result.metrics,
    edge_weight_semantics=manifest["edge_weight_semantics"],
)
assert "G_topology.gpickle" in manifest["files_created"]
display(reload_report)
display(Markdown(f"Final output directory: `{OUTPUT_DIR}`"))


## Synthetic validation tests


In [ ]:
synthetic_results = run_synthetic_separatrix_tests()
assert all(synthetic_results.values())
display(pd.Series(synthetic_results).to_frame("passed"))


## Final summary


In [ ]:
ann = transported.input_annotation_diagnostics
summary = f'''
### Reconstruction summary

- **method**: transported_separatrix_connectivity
- **selected q**: {config.hop_decay_q}
- **selected hop cutoff**: {config.max_soft_hops}
- **affinity mode**: {config.connectivity_affinity_mode}
- **unique separatrices / input (mean)**: {ann["unique_separatrix_id_count"].mean():.2f}
- **multi-ID edge fraction (mean)**: {ann["multi_id_edge_fraction"].mean():.3f}
- **nodes with no incident ID (sum)**: {int(ann["nodes_with_no_incident_id"].sum())}
- **disconnected per-ID subgraphs (sum)**: {int(ann["disconnected_per_id_subgraph_count"].sum())}
- **mean conditional direct-adjacency support**: {validation_result.metrics["mean_conditional_direct_adjacency_support"]}
- **mean conditional separatrix-connectivity support**: {validation_result.metrics["mean_conditional_separatrix_connectivity_support"]}
- **mean pair coverage**: {validation_result.metrics["mean_pair_coverage"]}
- **mean direct-adjacency evidence**: {validation_result.metrics["mean_direct_adjacency_evidence"]}
- **mean separatrix-connectivity evidence**: {validation_result.metrics["mean_separatrix_connectivity_evidence"]}
- **selected edges dominated by direct evidence**: {validation_result.metrics["selected_direct_dominated_fraction"]}
- **selected edges relying on soft extension**: {validation_result.metrics["selected_soft_dominated_fraction"]}
- **weakly supported nodes**: {validation_result.metrics["weakly_supported_node_count"]}
- **uncovered nodes**: {validation_result.metrics["uncovered_node_count"]}
- **final edge count**: {validation_result.metrics["edge_count"]}
- **component count**: {validation_result.metrics["component_count"]}
- **cycle rank**: {validation_result.metrics["cycle_rank"]}
- **length normalization materially changed result**: {length_changed}
- **transitive closure used**: False
- **C_center used for edge selection**: False
- **final graph**: `{OUTPUT_DIR / "G_topology.gpickle"}`
- **manifest**: `{OUTPUT_DIR / "manifest.json"}`

> The reconstructed topology is inferred from transported exact adjacency and soft connectivity along paths whose edges share one fixed recorded separatrix ID; overlapping separatrices are retained without merging their identities, and the barycenter relation is used only for optional post-hoc comparison.
'''
display(Markdown(summary))
